# Naive Bayes Using Scikit Learn.

---

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
data = pd.read_csv("spam_messages.csv")

In [3]:
data.drop(columns=['MessageID', 'CharacterCount', 'WordCount', 'HasURL', 'HasNumber', 'UppercaseRatio'], inplace=True)

## Feature Engineering:

In [4]:
def clean_text(feature):

    feature = feature.str.lower().str.strip().str.replace(r'https?://\S+|www\.\S+', 'URL', regex=True)
    feature = feature.str.replace(r'\b\d{7,15}\b', 'Phone-Number', regex=True)

    return feature

In [5]:
data['Message'] = clean_text(data['Message'])

In [6]:
data.dropna(inplace=True)

In [7]:
data.drop_duplicates(subset='Message', keep='first', inplace=True)

## Train-Test-Split

In [8]:
X = data['Message']
y = data['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [9]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Building Model Using Count-Vectorizer + Multinomial Naive Bayes:

In [10]:
count_vectorizer = CountVectorizer()

X_train_cv = count_vectorizer.fit_transform(X_train)
X_test_cv = count_vectorizer.transform(X_test)

In [11]:
nb = MultinomialNB()

start_time = time.time()
nb.fit(X_train_cv, y_train)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.01s


In [12]:
y_pred = nb.predict(X_test_cv)

print(f"Accuracy: {(accuracy_score(y_test, y_pred)*100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

print("\n")
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual Spam', 'Actual Ham'], columns=['Predicted Spam', 'Predicted Ham'])
matrix_df

Accuracy: 100.00%

Precision: 100.00%

Recall: 100.00%

F1 Score: 100.00%




,Predicted Spam,Predicted Ham
Actual Spam,19,0
Actual Ham,0,34


In [13]:
pipleine = Pipeline(steps=[
    ('vectorizer', CountVectorizer()),
    ('model', MultinomialNB())
])

print(f"Score: {np.mean(cross_val_score(estimator=pipleine, X=X, y=y, scoring='accuracy', cv=10, n_jobs=-1)) * 100:.2f}%")

Score: 99.23%


## Building Model Using TF-IDF + Multinomial Naive Bayes:

In [14]:
tf_idf_vectorizer = TfidfVectorizer()

X_train_tfidf = tf_idf_vectorizer.fit_transform(X_train)
X_test_tfidf = tf_idf_vectorizer.transform(X_test)

In [15]:
NB = MultinomialNB()

start_time = time.time()
NB.fit(X_train_tfidf, y_train)
end_time = time.time() - start_time

print("Model Trained Successfully!")
print(f"Time Taken for Training: {end_time:.2f}s")

Model Trained Successfully!
Time Taken for Training: 0.00s


In [16]:
y_pred = NB.predict(X_test_tfidf)

print(f"Accuracy: {(accuracy_score(y_test, y_pred)*100):.2f}%")

print(f"\nPrecision: {(precision_score(y_test, y_pred)*100):.2f}%")

print(f"\nRecall: {(recall_score(y_test, y_pred)*100):.2f}%")

print(f"\nF1 Score: {(f1_score(y_test, y_pred)*100):.2f}%")

print("\n")
matrix = confusion_matrix(y_test, y_pred)
matrix_df = pd.DataFrame(matrix, index=['Actual Spam', 'Actual Ham'], columns=['Predicted Spam', 'Predicted Ham'])
matrix_df

Accuracy: 100.00%

Precision: 100.00%

Recall: 100.00%

F1 Score: 100.00%




,Predicted Spam,Predicted Ham
Actual Spam,19,0
Actual Ham,0,34


In [17]:
pipleine = Pipeline(steps=[
    ('vectorizer', TfidfVectorizer()),
    ('model', MultinomialNB())
])

print(f"Score: {np.mean(cross_val_score(estimator=pipleine, X=X, y=y, scoring='accuracy', cv=10, n_jobs=-1)) * 100:.2f}%")

Score: 99.23%
